In [1]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive


def run(cmd, cwd=None):
    print("Running:", " ".join(map(str, cmd)))
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code:
        raise RuntimeError(
            f"Command failed with exit code {code}: "
            + " ".join(map(str, cmd))
        )


drive.mount("/content/drive")

WORKDIR = Path("/content/drive/MyDrive/Research/SSM_World_Models")
R2DREAMER_DIR = WORKDIR / "r2dreamer"
TDMPC2_DIR = WORKDIR / "tdmpc2"
DATA_DIR = WORKDIR / "data" / "dmc_expert"

R2DREAMER_REPO = "https://github.com/hanswalker/r2dreamer.git"
R2DREAMER_BRANCH = "mamba3-rssm-experiment"
TDMPC2_REPO = "https://github.com/nicklashansen/tdmpc2.git"

WORKDIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Colab starts with only this notebook. Clone the R2Dreamer repo that contains
# scripts/collect_dmc_expert_data.py, or update the Drive clone from the same
# GitHub source even if its default origin points somewhere else.
if not R2DREAMER_DIR.exists():
    run([
        "git",
        "clone",
        "--branch",
        R2DREAMER_BRANCH,
        R2DREAMER_REPO,
        str(R2DREAMER_DIR),
    ])
else:
    remotes = subprocess.run(
        ["git", "remote"],
        cwd=R2DREAMER_DIR,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.split()
    if "notebook-source" in remotes:
        run(["git", "remote", "set-url", "notebook-source", R2DREAMER_REPO], cwd=R2DREAMER_DIR)
    else:
        run(["git", "remote", "add", "notebook-source", R2DREAMER_REPO], cwd=R2DREAMER_DIR)
    run(["git", "fetch", "notebook-source", R2DREAMER_BRANCH], cwd=R2DREAMER_DIR)
    run(["git", "checkout", "-B", R2DREAMER_BRANCH, f"notebook-source/{R2DREAMER_BRANCH}"], cwd=R2DREAMER_DIR)

# TD-MPC2 is the expert policy repo. The collector imports its model code and
# downloads its released checkpoints from Hugging Face.
if not TDMPC2_DIR.exists():
    run(["git", "clone", TDMPC2_REPO, str(TDMPC2_DIR)])
else:
    run(["git", "pull", "--ff-only"], cwd=TDMPC2_DIR)

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "torch",
    "numpy==1.26.0",
    "pyyaml",
    "zarr<3",
    "huggingface_hub",
    "dm_control==1.0.28",
    "mujoco==3.3.0",
    "omegaconf",
    "hydra-core",
    "tensorboard>=2.20,<3",
    "gymnasium==1.2.0",
    "tensordict",
    "torchrl",
    "kornia",
    "termcolor",
    "tqdm",
    "pandas",
    "moviepy",
    "imageio",
    "imageio-ffmpeg",
    "h5py",
])

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TDMPC2_DIR"] = str(TDMPC2_DIR)
os.environ["DMC_EXPERT_DATA_DIR"] = str(DATA_DIR)

os.chdir(R2DREAMER_DIR)
if str(R2DREAMER_DIR) not in sys.path:
    sys.path.insert(0, str(R2DREAMER_DIR))

collector_candidates = [
    R2DREAMER_DIR / "scripts" / "collect_dmc_expert_data.py",
    R2DREAMER_DIR / "collect_dmc_expert_data.py",
]
COLLECTOR = next((path for path in collector_candidates if path.exists()), None)
if COLLECTOR is None:
    raise FileNotFoundError(
        "Missing collect_dmc_expert_data.py in the cloned R2Dreamer repo. "
        f"Checked: {collector_candidates}"
    )

R2DREAMER_COMMIT = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=R2DREAMER_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

import torch

print("R2Dreamer:", R2DREAMER_DIR)
print("R2Dreamer source:", R2DREAMER_REPO, R2DREAMER_BRANCH, R2DREAMER_COMMIT)
print("TD-MPC2:", TDMPC2_DIR)
print("Data:", DATA_DIR)
print("Collector:", COLLECTOR)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "TD-MPC2 requires CUDA. In Colab, choose Runtime > Change runtime type > GPU, "
        "then rerun the notebook."
    )


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running: git remote set-url notebook-source https://github.com/hanswalker/r2dreamer.git
Running: git fetch notebook-source mamba3-rssm-experiment
From https://github.com/hanswalker/r2dreamer
 * branch            mamba3-rssm-experiment -> FETCH_HEAD
Running: git checkout -B mamba3-rssm-experiment notebook-source/mamba3-rssm-experiment
Reset branch 'mamba3-rssm-experiment'
M	runs/atari.sh
M	runs/crafter.sh
M	runs/dmc.sh
M	runs/dmc_subtle.sh
M	runs/isaaclab.sh
M	runs/memorymaze.sh
M	runs/metaworld.sh
Branch 'mamba3-rssm-experiment' set up to track remote branch 'mamba3-rssm-experiment' from 'notebook-source'.
Your branch is up to date with 'notebook-source/mamba3-rssm-experiment'.
Running: git pull --ff-only
Already up to date.
Running: /usr/bin/python3 -m pip install -q torch numpy==1.26.0 pyyaml zarr<3 huggingface_hub dm_control==1.0.28 mujoco==3.3.0 omegaconf

In [ ]:
CONFIG_PATH = R2DREAMER_DIR / "configs" / "dmc_expert_small_train.yaml"

print("Config:", CONFIG_PATH)
print(CONFIG_PATH.read_text())
run([sys.executable, str(COLLECTOR), "--help"])
run([sys.executable, str(COLLECTOR), "--config", str(CONFIG_PATH)])


In [2]:
from pathlib import Path
import subprocess
import sys
import time
import textwrap

import zarr
from IPython.display import clear_output

parallel_tasks = [
    "walker/walk",       # resumes the first run
    "cartpole/swingup",
]

NUM_EPISODES = 500
CHECKPOINT_SEED = 1
SEED = 1
ACTION_REPEAT = 2
MAX_EPISODE_STEPS = 1000
SAVE_IMAGES = True
IMAGE_SIZE = 64
REFRESH_SECONDS = 10
RECENT_LOG_LINES = 80

PARALLEL_ROOT = DATA_DIR / "parallel_runs"
CONFIG_DIR = PARALLEL_ROOT / "configs"
LOG_DIR = PARALLEL_ROOT / "logs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)


def task_store_name(task):
    if task == "ball_in_cup/catch":
        return "cup_catch.zarr"
    return task.replace("/", "_").replace("-", "_").replace("__", "_") + ".zarr"


def read_progress(out_dir, task):
    store_path = out_dir / task_store_name(task)
    if not store_path.exists():
        return 0, 0, "not created"
    try:
        root = zarr.open(str(store_path), mode="r")
        episodes = int(root["episode_length"].shape[0]) if "episode_length" in root else 0
        rows = int(root["obs"].shape[0]) if "obs" in root else 0
        return episodes, rows, str(store_path)
    except Exception as exc:
        return 0, 0, f"read pending: {type(exc).__name__}"


def read_new_log_lines(log_path, offset):
    if not log_path.exists():
        return offset, []
    with log_path.open("r", encoding="utf-8", errors="replace") as f:
        f.seek(offset)
        lines = f.readlines()
        offset = f.tell()
    return offset, lines


processes = []
log_offsets = {}
recent_lines = []

for task in parallel_tasks:
    slug = task.replace("/", "_")

    # Keep walker/walk in the original DATA_DIR so it resumes the existing first run.
    # Put the other tasks in separate folders to avoid manifest write races.
    out_dir = DATA_DIR if task == "walker/walk" else PARALLEL_ROOT / slug
    out_dir.mkdir(parents=True, exist_ok=True)

    config_path = CONFIG_DIR / f"{slug}.yaml"
    log_path = LOG_DIR / f"{slug}.log"

    config_path.write_text(
        textwrap.dedent(f"""
        tdmpc2_root: {TDMPC2_DIR}
        output_dir: {out_dir}

        tasks:
          - {task}

        num_episodes: {NUM_EPISODES}
        checkpoint_seed: {CHECKPOINT_SEED}
        seed: {SEED}

        action_repeat: {ACTION_REPEAT}
        max_episode_steps: {MAX_EPISODE_STEPS}

        save_images: {str(SAVE_IMAGES).lower()}
        image_size: {IMAGE_SIZE}

        resume: true
        """).strip() + "\n",
        encoding="utf-8",
    )

    log_file = log_path.open("w", buffering=1, encoding="utf-8")
    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        [sys.executable, "-u", str(COLLECTOR), "--config", str(config_path)],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )

    processes.append((task, proc, log_file, log_path, out_dir, config_path))
    log_offsets[task] = 0
    recent_lines.append(f"[{task}] started pid={proc.pid}, output={out_dir}, log={log_path}")


while True:
    all_done = True
    statuses = []

    for task, proc, _, log_path, out_dir, _ in processes:
        log_offsets[task], lines = read_new_log_lines(log_path, log_offsets[task])
        for line in lines:
            line = line.rstrip()
            if line:
                recent_lines.append(f"[{task}] {line}")
        recent_lines = recent_lines[-RECENT_LOG_LINES:]

        code = proc.poll()
        if code is None:
            all_done = False
            state = "running"
        elif code == 0:
            state = "done"
        else:
            state = f"failed({code})"

        episodes, rows, store = read_progress(out_dir, task)
        statuses.append((task, state, episodes, rows, store, log_path))

    clear_output(wait=True)
    print(f"Live collector progress | refresh={REFRESH_SECONDS}s | target={NUM_EPISODES} episodes per task")
    print("-" * 120)
    print(f"{'task':<22} {'state':<12} {'episodes':>10} {'rows':>10}  store/log")
    for task, state, episodes, rows, store, log_path in statuses:
        print(f"{task:<22} {state:<12} {episodes:>4}/{NUM_EPISODES:<5} {rows:>10}  {store}")
        print(f"{'':<48} log: {log_path}")

    print("\nRecent log lines")
    print("-" * 120)
    print("\n".join(recent_lines[-RECENT_LOG_LINES:]) if recent_lines else "No log lines yet.")

    if all_done:
        break
    time.sleep(REFRESH_SECONDS)

failed = []
for task, proc, log_file, log_path, out_dir, config_path in processes:
    log_file.close()
    if proc.returncode != 0:
        failed.append((task, log_path))

if failed:
    print("\nFailures:")
    for task, log_path in failed:
        print(f"\n--- {task} last log lines ---")
        lines = log_path.read_text(errors="replace").splitlines()
        print("\n".join(lines[-120:]))
    raise RuntimeError(f"{len(failed)} collectors failed")

print("\nAll collectors finished successfully.")


Live collector progress | refresh=10s | target=500 episodes per task
------------------------------------------------------------------------------------------------------------------------
task                   state          episodes       rows  store/log
walker/walk            done          500/500       250000  /content/drive/MyDrive/Research/SSM_World_Models/data/dmc_expert/walker_walk.zarr
                                                 log: /content/drive/MyDrive/Research/SSM_World_Models/data/dmc_expert/parallel_runs/logs/walker_walk.log
cartpole/swingup       done          500/500       250500  /content/drive/MyDrive/Research/SSM_World_Models/data/dmc_expert/parallel_runs/cartpole_swingup/cartpole_swingup.zarr
                                                 log: /content/drive/MyDrive/Research/SSM_World_Models/data/dmc_expert/parallel_runs/logs/cartpole_swingup.log

Recent log lines
--------------------------------------------------------------------------------------------

In [ ]:
# Offline training from the collected expert zarr dataset.
# Add "size_small_mamba3" to TRAIN_MODELS after Mamba3 is installed.

TRAIN_MODELS = ["size_small_gru"]
TRAIN_TASK = "walker/walk"
TRAIN_STORE = DATA_DIR / "walker_walk.zarr"

if not TRAIN_STORE.exists():
    raise FileNotFoundError(f"Missing training dataset: {TRAIN_STORE}")

os.environ["DMC_EXPERT_DATA_DIR"] = str(DATA_DIR)
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

for train_model in TRAIN_MODELS:
    train_logdir = WORKDIR / "runs" / f"offline_{train_model}_walker_walk"
    run([
        sys.executable,
        "-u",
        "train.py",
        "--config-name",
        "offline_dmc_expert",
        f"model={train_model}",
        f"offline.data_path={TRAIN_STORE}",
        "env.task=dmc_walker_walk",
        f"logdir={train_logdir}",
    ], cwd=R2DREAMER_DIR)
    print("Training logdir:", train_logdir)
